# JAX/Flax Qwen2 Verilog Training on Kaggle TPU v3-8

This notebook trains a fine-tuned Qwen2 model for Verilog code generation using JAX/Flax on 8 TPU cores.

**Accelerator**: TPU VM v3-8 (8 cores)
**Model**: Qwen2.5-Coder-7B-Instruct (or 14B with SPMD)
**Precision**: bfloat16 (TPU native)

## Cell 1: Verify TPU and install dependencies

In [ ]:
import os
import subprocess
import sys

# Install JAX TPU build BEFORE importing jax.
# Kaggle TPU images may ship CPU-only jax; this provides libtpu.so.
print('Installing JAX TPU stack...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'jax[tpu]',
    '-f', 'https://storage.googleapis.com/jax-releases/libtpu_releases.html'
], check=False)

# Install additional packages
packages = [
    'flax>=0.8.0',
    'optax>=0.2.0',
    'transformers>=4.40.0',
    'datasets>=2.14.0',
    'huggingface-hub>=0.20.0',
    'sentencepiece>=0.1.99',
    'safetensors>=0.4.0',
    'torch>=2.0.0',
    'protobuf>=3.20.0,<5.0.0',
    'jinja2>=3.1.0',
]
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', pkg], check=False)

# Verify JAX sees TPU
import jax
print(f'JAX version: {jax.__version__}')
try:
    devices = jax.devices('tpu')
except RuntimeError as e:
    print('✗ TPU backend not available.')
    print(e)
    print('ACTION: In Kaggle notebook settings, select Accelerator = TPU VM v3-8, then Save & Restart.')
    raise

print(f'TPU devices: {len(devices)}')
for d in devices:
    print(f'  {d}')
if len(devices) != 8:
    raise RuntimeError(f'Expected 8 TPU devices, got {len(devices)}')

print('\n✓ Dependencies installed and TPU ready')


## Cell 2: Authenticate with HuggingFace

In [ ]:
from huggingface_hub import login

HF_TOKEN = None

# Try Kaggle Secrets first
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✓ HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print(f"Kaggle Secrets not available: {e}")

# Fallback to env var
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from environment")

if not HF_TOKEN:
    raise ValueError("Please set HF_TOKEN in Kaggle Secrets or environment")

login(token=HF_TOKEN)
print("✓ HuggingFace authenticated")

## Cell 3: Write source files to disk

In [ ]:
# Write all Python modules directly into the notebook
# (so we don't need to upload files separately)

import os
os.makedirs("/kaggle/working/jax_flax_port", exist_ok=True)

# config.py
config_py = '''"""Centralized configuration for JAX/Flax Verilog training."""
from dataclasses import dataclass
from typing import Optional

@dataclass
class ModelConfig:
    model_name: str = "Qwen/Qwen2.5-Coder-7B-Instruct"
    hub_model_id: str = "Pablo-Flores-Mollinedo/verilog-qwen-14b-sota"
    vocab_size: int = 152064
    hidden_size: int = 3584
    intermediate_size: int = 18944
    num_hidden_layers: int = 28
    num_attention_heads: int = 28
    num_key_value_heads: int = 4
    max_position_embeddings: int = 32768
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1_000_000.0
    use_sliding_window: bool = False
    sliding_window: int = 131072
    tie_word_embeddings: bool = False
    dtype: str = "bfloat16"
    @property
    def head_dim(self) -> int:
        return self.hidden_size // self.num_attention_heads

@dataclass
class LoRAConfig:
    r: int = 64
    alpha: int = 128
    dropout: float = 0.05
    target_modules: tuple = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

@dataclass
class TrainConfig:
    output_dir: str = "/kaggle/working/checkpoints"
    max_seq_length: int = 2048
    batch_size: int = 1
    grad_accum: int = 4
    learning_rate: float = 2e-4
    min_lr: float = 0.0
    warmup_steps: int = 50
    num_epochs: int = 3
    weight_decay: float = 0.001
    max_grad_norm: float = 0.3
    seed: int = 42
    train_path: str = "/kaggle/input/verilog-curated-dataset/train.jsonl"
    eval_path: str = "/kaggle/input/verilog-curated-dataset/eval.jsonl"
    working_dataset_dir: str = "/kaggle/working/verilog-curated-dataset"
    save_steps: int = 100
    save_total_limit: int = 3
    log_every: int = 10

MODEL_7B = ModelConfig()
MODEL_14B = ModelConfig(
    model_name="Qwen/Qwen2.5-Coder-14B-Instruct",
    hidden_size=5120,
    intermediate_size=13824,
    num_hidden_layers=48,
    num_attention_heads=40,
    num_key_value_heads=8,
)
'''

with open("/kaggle/working/jax_flax_port/config.py", "w") as f:
    f.write(config_py)

print("✓ config.py written")

## Cell 4: Write model_flax.py (simplified inline)

For brevity, we inline a minimal Flax model here. For production, use the full `model_flax.py` from the repo.

In [ ]:
# Since model_flax.py is too large to inline comfortably,
# we will git-clone or download the source files from your GitHub repo.

import subprocess
import sys

# Clone the repo to get the full jax_flax_port source
repo_url = "https://github.com/pabloflores465/verilog-llm.git"
clone_dir = "/kaggle/working/verilog-llm"

if not os.path.exists(clone_dir):
    print("Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1", repo_url, clone_dir], check=True)
    print("✓ Repo cloned")
else:
    print("✓ Repo already present")

# Symlink or copy the jax_flax_port files
import shutil
src = "/kaggle/working/verilog-llm/jax_flax_port"
dst = "/kaggle/working/jax_flax_port"
for f in os.listdir(src):
    s = os.path.join(src, f)
    d = os.path.join(dst, f)
    if os.path.isfile(s):
        shutil.copy2(s, d)
print("✓ Source files copied")

## Cell 5: Convert PyTorch weights to JAX (one-time)

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/jax_flax_port")

from convert_weights import load_pytorch_state_dict, convert_to_flax_params, save_jax_checkpoint
from config import MODEL_7B

output_path = "/kaggle/working/qwen_jax_weights.npz"

if os.path.exists(output_path):
    print(f"✓ JAX weights already exist: {output_path}")
else:
    print("Converting PyTorch weights to JAX. This takes ~5 minutes...")
    state_dict = load_pytorch_state_dict(MODEL_7B.model_name)
    flax_params = convert_to_flax_params(state_dict, MODEL_7B)
    save_jax_checkpoint(flax_params, output_path)
    print("✓ Conversion complete")

## Cell 6: Launch training

In [ ]:
# Run the 7B data-parallel training script
import subprocess
cmd = [
    sys.executable,
    "/kaggle/working/jax_flax_port/train_dp.py",
    "--model", "7b",
    "--epochs", "3",
    "--max_seq_length", "2048",
    "--batch_size", "1",
    "--grad_accum", "4",
    "--lr", "2e-4",
    "--save_steps", "100",
    "--from_jax_weights", "/kaggle/working/qwen_jax_weights.npz",
]

print("Starting training...")
print(" ".join(cmd))
result = subprocess.run(cmd)
sys.exit(result.returncode)
